In [2]:
import sys
!{sys.executable} -m pip install -q pandas pyarrow azure-storage-blob


[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: pip3 install --upgrade pip


In [3]:
from azure.storage.blob import ContainerClient
from io import BytesIO
import pandas as pd

# Джерело (тільки читання)
SOURCE_SAS = "ВСТАВТЕ_SAS_ДЖЕРЕЛА"

# Озеро (читання + запис)
LAKE_SAS = "ВСТАВТЕ_SAS_ОЗЕРА"

source = ContainerClient.from_container_url(SOURCE_SAS)
lake   = ContainerClient.from_container_url(LAKE_SAS)

STUDENT = "krotova_iryna"

In [4]:
# читання JSON із джерела
def read_json(name):
    raw = source.download_blob(name).readall()
    return pd.read_json(BytesIO(raw))

# читання Parquet зі свого префікса в озері
def read_parquet(path):
    raw = lake.download_blob(f"{STUDENT}/{path}").readall()
    return pd.read_parquet(BytesIO(raw))

# запис DataFrame у Parquet у свій префікс
def write_parquet(df, path):
    buf = BytesIO()
    df.to_parquet(buf, index=False)
    buf.seek(0)
    lake.upload_blob(name=f"{STUDENT}/{path}", data=buf.getvalue(), overwrite=True)

In [5]:
shops_src = read_json("shops.json")
products_src = read_json("products.json")
print("shops:", len(shops_src))
print("products:", len(products_src))

shops: 5
products: 12


In [6]:
parts = [read_json(f"orders_2026_0{m}.json") for m in (1, 2, 3)]
bronze_orders = pd.concat(parts, ignore_index=True)
print(len(bronze_orders))

980


In [7]:
write_parquet(bronze_orders, "bronze/orders.parquet")
write_parquet(shops_src, "bronze/shops.parquet")
write_parquet(products_src, "bronze/products.parquet")

In [8]:
prefix = f"{STUDENT}/bronze/"
for blob in lake.list_blobs(name_starts_with=prefix):
    if "/" not in blob.name[len(prefix):]:   # частина шляху після bronze/ без "/" — це файл, а не підпапка
        print(blob.name, blob.size)

krotova_iryna/bronze/orders.parquet 18580
krotova_iryna/bronze/products.parquet 3573
krotova_iryna/bronze/shops.parquet 3974


In [9]:
silver = read_parquet("bronze/orders.parquet")

print("повних дублікатів:", silver.duplicated().sum())
print("унікальних shop_code:", silver["shop_code"].nunique())
print("унікальних payment_type:", silver["payment_type"].nunique())
print("порожніх qty:", silver["qty"].isna().sum())


print(silver["shop_code"].unique())
print(silver["payment_type"].unique())

повних дублікатів: 20
унікальних shop_code: 10
унікальних payment_type: 4
порожніх qty: 36
<ArrowStringArray>
[   'S04',    'S05',    'S01',    'S03',    'S02', '  s03 ', '  s05 ',
 '  s04 ', '  s01 ', '  s02 ']
Length: 10, dtype: str
<ArrowStringArray>
['Card', 'Cash', 'CASH', 'CARD']
Length: 4, dtype: str


In [10]:
silver = silver.drop_duplicates().reset_index(drop=True)
print(len(silver))

960


In [11]:
silver["shop_code"] = silver["shop_code"].str.strip().str.upper()
print(sorted(silver["shop_code"].unique()))

['S01', 'S02', 'S03', 'S04', 'S05']


In [12]:
silver["payment_type"] = silver["payment_type"].str.strip().str.capitalize()
print(sorted(silver["payment_type"].unique()))

['Card', 'Cash']


In [13]:
silver["qty"] = silver["qty"].fillna(1).astype(int)
print(silver["qty"].dtype, silver["qty"].isna().sum())

int64 0


In [14]:
silver["order_ts"] = pd.to_datetime(silver["order_ts"])
silver["order_date"] = silver["order_ts"].dt.date
print(silver[["order_ts", "order_date"]].head(3))
print(silver["order_ts"].dtype)

             order_ts  order_date
0 2026-01-27 12:05:00  2026-01-27
1 2026-01-03 09:30:00  2026-01-03
2 2026-01-30 11:35:00  2026-01-30
datetime64[us]


In [15]:
silver["amount"] = silver["qty"] * silver["unit_price"]
print(silver[["qty", "unit_price", "amount"]].head(3))
print("сума amount:", silver["amount"].sum())

   qty  unit_price  amount
0    2          80     160
1    2          80     160
2    1          95      95
сума amount: 90320


In [16]:
write_parquet(silver, "silver/orders.parquet")

write_parquet(read_parquet("bronze/shops.parquet"), "silver/shops.parquet")
write_parquet(read_parquet("bronze/products.parquet"), "silver/products.parquet")

In [17]:
prefix = f"{STUDENT}/silver/"
for blob in lake.list_blobs(name_starts_with=prefix):
    print(blob.name, blob.size)

print(read_parquet("silver/orders.parquet").dtypes)

krotova_iryna/silver/orders.parquet 21695
krotova_iryna/silver/products.parquet 3573
krotova_iryna/silver/readings/year=2026/month=01/day=07/part.parquet 4364
krotova_iryna/silver/readings/year=2026/month=02/day=07/part.parquet 4364
krotova_iryna/silver/readings/year=2026/month=04/day=07/part.parquet 4364
krotova_iryna/silver/readings/year=2026/month=07/day=01/part.parquet 4458
krotova_iryna/silver/readings/year=2026/month=07/day=02/part.parquet 4398
krotova_iryna/silver/readings/year=2026/month=07/day=03/part.parquet 4425
krotova_iryna/silver/readings/year=2026/month=07/day=04/part.parquet 4441
krotova_iryna/silver/readings/year=2026/month=07/day=05/part.parquet 4364
krotova_iryna/silver/sensors/sensors.parquet 2773
krotova_iryna/silver/shops.parquet 3974
krotova_iryna/silver/stations/stations.parquet 3371
order_id                   str
order_ts        datetime64[us]
shop_code                  str
product_code               str
qty                      int64
unit_price               i

In [18]:
prefix = f"{STUDENT}/silver/"
for blob in lake.list_blobs(name_starts_with=prefix):
    if "/" not in blob.name[len(prefix):]:
        print(blob.name, blob.size)

print(read_parquet("silver/orders.parquet").dtypes)

krotova_iryna/silver/orders.parquet 21695
krotova_iryna/silver/products.parquet 3573
krotova_iryna/silver/shops.parquet 3974
order_id                   str
order_ts        datetime64[us]
shop_code                  str
product_code               str
qty                      int64
unit_price               int64
payment_type               str
order_date              object
amount                   int64
dtype: object


In [19]:
silver_orders   = read_parquet("silver/orders.parquet")
silver_shops    = read_parquet("silver/shops.parquet")
silver_products = read_parquet("silver/products.parquet")
print(len(silver_orders), len(silver_shops), len(silver_products))

960 5 12


In [20]:
dim_shop = silver_shops.sort_values("shop_code").reset_index(drop=True)
dim_shop.insert(0, "shop_key", range(1, len(dim_shop) + 1))
dim_shop

,shop_key,shop_code,shop_name,city,district,opened_date,seats
0,1,S01,Kava Hub,Kyiv,Podil,2019-04-15,24
1,2,S02,Zerno,Kyiv,Pechersk,2021-09-01,16
2,3,S03,Rynok Coffee,Lviv,Center,2018-06-20,30
3,4,S04,Bereg,Odesa,Arkadia,2022-05-10,40
4,5,S05,Steam,Kharkiv,Center,2023-02-01,12


In [21]:
dim_product = silver_products.sort_values("product_code").reset_index(drop=True)
dim_product.insert(0, "product_key", range(1, len(dim_product) + 1))
dim_product

,product_key,product_code,product_name,category,size,price
0,1,P01,Espresso,Coffee,S,45
1,2,P02,Americano,Coffee,M,55
2,3,P03,Cappuccino,Coffee,M,70
3,4,P04,Latte,Coffee,L,80
4,5,P05,Flat White,Coffee,M,75
5,6,P06,Green Tea,Tea,M,50
6,7,P07,Herbal Tea,Tea,M,50
7,8,P08,Croissant,Bakery,-,60
8,9,P09,Cinnamon Bun,Bakery,-,65
9,10,P10,Sandwich,Bakery,-,110


In [22]:
fact_sales = (
    silver_orders
    .merge(dim_shop[["shop_key", "shop_code"]],
           on="shop_code", how="left", validate="many_to_one")
    .merge(dim_product[["product_key", "product_code"]],
           on="product_code", how="left", validate="many_to_one")
    [["order_id", "shop_key", "product_key", "order_date",
      "qty", "unit_price", "amount", "payment_type"]]
)
print(len(fact_sales))
print("пропусків у ключах:", fact_sales[["shop_key", "product_key"]].isna().sum().sum())
fact_sales.head()

960
пропусків у ключах: 0


,order_id,shop_key,product_key,order_date,qty,unit_price,amount,payment_type
0,O500259,4,4,2026-01-27,2,80,160,Card
1,O500252,5,4,2026-01-03,2,80,160,Card
2,O500257,4,11,2026-01-30,1,95,95,Cash
3,O500127,1,10,2026-01-03,3,110,330,Card
4,O500225,3,4,2026-01-28,3,80,240,Cash


In [23]:
fact_daily_sales = (
    fact_sales
    .groupby(["shop_key", "order_date"], as_index=False)
    .agg(orders_count=("order_id", "nunique"),
         items_sold=("qty", "sum"),
         revenue=("amount", "sum"))
)
print(len(fact_daily_sales))
fact_daily_sales.head()

396


,shop_key,order_date,orders_count,items_sold,revenue
0,1,2026-01-01,2,3,235
1,1,2026-01-02,1,1,95
2,1,2026-01-03,6,9,785
3,1,2026-01-04,2,3,225
4,1,2026-01-05,2,3,170


In [24]:
write_parquet(dim_shop, "gold/dim_shop.parquet")
write_parquet(dim_product, "gold/dim_product.parquet")
write_parquet(fact_sales, "gold/fact_sales.parquet")
write_parquet(fact_daily_sales, "gold/fact_daily_sales.parquet")

prefix = f"{STUDENT}/gold/"
for blob in lake.list_blobs(name_starts_with=prefix):
    if "/" not in blob.name[len(prefix):]:
        print(blob.name, blob.size)

krotova_iryna/gold/dim_product.parquet 4205
krotova_iryna/gold/dim_shop.parquet 4560
krotova_iryna/gold/fact_daily_sales.parquet 5501
krotova_iryna/gold/fact_sales.parquet 13745


In [25]:
dim_shop         = read_parquet("gold/dim_shop.parquet")
dim_product      = read_parquet("gold/dim_product.parquet")
fact_sales       = read_parquet("gold/fact_sales.parquet")
fact_daily_sales = read_parquet("gold/fact_daily_sales.parquet")

# [умова] очікувано: 5, 12, 960, 396
for name, df in [("dim_shop", dim_shop), ("dim_product", dim_product),
                 ("fact_sales", fact_sales), ("fact_daily_sales", fact_daily_sales)]:
    print(name, len(df))

dim_shop 5
dim_product 12
fact_sales 960
fact_daily_sales 396


In [26]:
print(fact_sales["shop_key"].isin(dim_shop["shop_key"]).all())
print(fact_sales["product_key"].isin(dim_product["product_key"]).all())

True
True


In [27]:
print(fact_sales["amount"].sum() == fact_daily_sales["revenue"].sum())

True


In [28]:
revenue_by_category = (
    fact_sales
    .merge(dim_product[["product_key", "category"]], on="product_key")
    .groupby("category")["amount"].sum()
    .sort_values(ascending=False)
)
revenue_by_category

category
Coffee     49915
Bakery     19955
Dessert    14850
Tea         5600
Name: amount, dtype: int64

In [29]:
revenue_by_shop = (
    fact_sales
    .merge(dim_shop[["shop_key", "shop_name"]], on="shop_key")
    .groupby("shop_name")["amount"].sum()
    .sort_values(ascending=False)
)
revenue_by_shop

shop_name
Rynok Coffee    20075
Kava Hub        19290
Steam           18275
Bereg           16750
Zerno           15930
Name: amount, dtype: int64

In [30]:
files = []
for layer in ("bronze", "silver", "gold"):
    prefix = f"{STUDENT}/{layer}/"
    for blob in lake.list_blobs(name_starts_with=prefix):
        if "/" not in blob.name[len(prefix):]:
            files.append(blob.name)

print(len(files))   # очікувано 10
for f in files:
    print(f)

10
krotova_iryna/bronze/orders.parquet
krotova_iryna/bronze/products.parquet
krotova_iryna/bronze/shops.parquet
krotova_iryna/silver/orders.parquet
krotova_iryna/silver/products.parquet
krotova_iryna/silver/shops.parquet
krotova_iryna/gold/dim_product.parquet
krotova_iryna/gold/dim_shop.parquet
krotova_iryna/gold/fact_daily_sales.parquet
krotova_iryna/gold/fact_sales.parquet
